# 🧪 AiStock 性能基准测试 (Plotly 可视化版)

## 测试目标 + 交互式性能分析可视化
- ✅ 缓存性能：LRU 策略 + TTL 配置（命中率热力图 + 容量对比）
- ✅ 数据加载：数据库查询 + API 调用（耗时箱线图 + 吞吐量对比）
- ✅ 计算引擎：三维加权计算（并行加速比 + 内存使用曲线）
- ✅ 系统吞吐：端到端流程（吞吐量曲线 + 资源监控）

## Plotly 高级特性：箱线图、热力图、3D 图表、实时刷新模拟

In [ ]:
# 环境设置 + Plotly 性能分析配置 + 中文支持准备 + 性能监控工具
import sys
from pathlib import Path
import pandas as pd
import numpy as np
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import time
import tracemalloc

# Plotly 主题 + 中文配置（可选：安装中文字体后启用）
import plotly.io as pio
pio.templates.default = "plotly_white"

# 添加项目根目录到路径 + 性能测试工具函数（演示用）
project_root = Path.cwd().parent
sys.path.insert(0, str(project_root))

# 性能测试辅助函数（模拟）
def run_performance_test(func, *args, iterations=10, **kwargs):
    """运行性能测试并返回统计结果"""
    times = []
    for _ in range(iterations):
        start = time.time()
        result = func(*args, **kwargs)
        end = time.time()
        times.append((end - start) * 1000)  # 转换为毫秒
    
    return {
        'mean': np.mean(times),
        'std': np.std(times),
        'min': np.min(times),
        'max': np.max(times),
        'p95': np.percentile(times, 95),
        'samples': times
    }

print("✅ 环境设置完成 | Plotly 性能分析已启用")

In [ ]:
# 导入性能测试相关模块 + 模拟测试函数（演示）
from base_services.config_service import ConfigService
from base_services.cache_service import CacheService
from base_services.database_service import DatabaseService
from dynamic_price_system.data.data_loader import DataLoader
from dynamic_price_system.core.dynamic_price_engine import DynamicPriceEngine
from config.global_settings import DATABASE_MAIN_URL, DB_POOL_CONFIG

# 模拟缓存性能测试函数（演示）
def test_cache_performance(cache_size, ttl=3600, operations=1000):
    """测试不同缓存配置的性能"""
    cache = CacheService(max_size=cache_size, ttl=ttl)
    
    # 写入测试（模拟）
    write_times = []
    for i in range(operations):
        start = time.time()
        cache.set(f'key_{i}', {'value': np.random.random()})
        write_times.append((time.time() - start) * 1000)
    
    # 读取测试（模拟）
    read_times = []
    for i in range(operations):
        start = time.time()
        cache.get(f'key_{i}' if i % 2 == 0 else f'nonexistent_{i}')
        read_times.append((time.time() - start) * 1000)
    
    return {
        'cache_size': cache_size,
        'write_mean': np.mean(write_times),
        'read_mean': np.mean(read_times),
        'hit_rate': cache.get_hit_rate(),
        'throughput': operations / (np.sum(write_times) + np.sum(read_times)) * 1000
    }

print("✅ 性能测试模块导入成功")

## 1️⃣ 缓存性能测试 + 交互式对比图

In [ ]:
# 执行缓存性能测试（模拟）+ Plotly 多维度对比图（交互式）
cache_sizes = [100, 500, 1000, 2000, 5000]
cache_results = []

for size in cache_sizes:
    result = test_cache_performance(cache_size=size, operations=500)
    cache_results.append(result)
    print(f"✅ 缓存容量 {size}: 命中率 {result['hit_rate']:.1%}, 吞吐量 {result['throughput']:.0f} ops/s")

cache_df = pd.DataFrame(cache_results)

# 创建交互式缓存性能对比图（多指标 + 双轴）
fig = make_subplots(
    rows=1, cols=2,
    subplot_titles=['命中率 vs 容量', '吞吐量趋势'],
    specs=[[{'secondary_y': False}, {'secondary_y': True}]]
)

# 左图：命中率随容量变化（折线图 + 参考线）
fig.add_trace(go.Scatter(
    x=cache_df['cache_size'],
    y=cache_df['hit_rate'],
    mode='lines+markers',
    name='缓存命中率',
    line=dict(color='blue', width=3),
    marker=dict(size=8)
), row=1, col=1)

# 添加目标命中率参考线（80%）
fig.add_hline(
    y=0.80,
    line_dash='dash',
    line_color='green',
    annotation_text='目标: 80%',
    annotation_position='top right',
    row=1, col=1
)

# 右图：吞吐量随容量变化（柱状图 + 折线图组合）
fig.add_trace(go.Bar(
    x=cache_df['cache_size'],
    y=cache_df['throughput'],
    name='吞吐量',
    marker_color='skyblue',
    text=cache_df['throughput'].apply(lambda x: f'{x:.0f}'),
    textposition='auto'
), row=1, col=2)

# 添加读写时间对比（次坐标轴）
fig.add_trace(go.Scatter(
    x=cache_df['cache_size'],
    y=cache_df['write_mean'] + cache_df['read_mean'],
    name='平均读写时间',
    mode='lines+markers',
    line=dict(color='red', width=2),
    marker=dict(size=6),
    yaxis='y2'
), row=1, col=2)

# 更新布局 + 交互式设置（双轴 + 悬停提示）
fig.update_layout(
    title='⚡ 缓存性能基准测试',
    height=400,
    hovermode='x unified',
    legend=dict(orientation='h', yanchor='bottom', y=-0.3, xanchor='center', x=0.5)
)

# 更新坐标轴标签 + 双轴配置 + 对数坐标（容量）
fig.update_xaxes(title_text='缓存容量 (条)', type='log', row=1, col=1)
fig.update_yaxes(title_text='命中率', tickformat='.0%', row=1, col=1)
fig.update_xaxes(title_text='缓存容量 (条)', type='log', row=1, col=2)
fig.update_yaxes(title_text='吞吐量 (ops/s)', row=1, col=2)
fig.update_yaxes(title_text='读写时间 (ms)', secondary_y=True, row=1, col=2)

fig.show()

In [ ]:
# Plotly 缓存命中率热力图（容量×请求分布）
# 模拟不同容量 + 不同请求模式下的命中率矩阵（用于热力图）
capacities = [100, 500, 1000, 2000]
request_patterns = ['均匀', '热点集中', '随机', '时间局部性']

# 生成模拟命中率矩阵（实际应通过压力测试获取）
np.random.seed(42)
hit_rate_matrix = np.zeros((len(capacities), len(request_patterns)))

for i, cap in enumerate(capacities):
    for j, pattern in enumerate(request_patterns):
        # 模拟不同模式下的命中率（容量越大命中率越高，热点模式命中率更高）
        base_rate = min(0.95, 0.5 + np.log10(cap) * 0.15)
        pattern_bonus = {'均匀': 0, '热点集中': 0.1, '随机': -0.05, '时间局部性': 0.05}[pattern]
        hit_rate_matrix[i, j] = np.clip(base_rate + pattern_bonus + np.random.normal(0, 0.03), 0.3, 0.99)

# 创建交互式热力图（容量×请求模式）
fig = px.imshow(
    hit_rate_matrix,
    x=request_patterns,
    y=[f'{cap}条' for cap in capacities],
    text_auto='.1%',
    aspect='auto',
    color_continuous_scale='RdYlGn',
    title='🔥 缓存命中率热力图（容量×请求模式）',
    labels={'x': '请求模式', 'y': '缓存容量', 'color': '命中率'}
)

# 添加颜色参考说明（命中率区间）
fig.add_annotation(
    x=-0.5, y=-0.8,
    text='🟢 >90%: 优秀  |  🟡 70-90%: 良好  |  🟠 50-70%: 一般  |  🔴 <50%: 需优化',
    showarrow=False,
    font=dict(size=10),
    bgcolor='white',
    bordercolor='gray',
    borderwidth=1,
    xref='paper', yref='paper'
)

fig.update_layout(
    height=400,
    margin=dict(l=20, r=20, t=50, b=40),
    coloraxis_colorbar=dict(title='命中率', tickformat='.0%')
)

fig.show()

## 2️⃣ 数据加载性能 + 交互式箱线图

In [ ]:
# 数据加载性能测试（模拟）+ Plotly 箱线图 + 散点图组合（交互式）
# 模拟不同数据源的加载耗时（实际应通过真实测试获取）
data_sources = ['数据库查询', 'TDX 接口', '外部 API', '缓存命中', '缓存未命中']
load_times = {
    '数据库查询': np.random.lognormal(2.0, 0.5, 100),  # 对数正态分布模拟真实耗时
    'TDX 接口': np.random.lognormal(2.2, 0.6, 100),
    '外部 API': np.random.lognormal(2.5, 0.7, 100),
    '缓存命中': np.random.lognormal(1.0, 0.3, 100),
    '缓存未命中': np.random.lognormal(2.3, 0.6, 100)
}

# 准备箱线图数据（长格式）
box_data = []
for source, times in load_times.items():
    for t in times:
        box_data.append({'source': source, 'time_ms': t})

box_df = pd.DataFrame(box_data)

# 创建交互式箱线图（带散点 + 统计标注）
fig = px.box(
    box_df,
    x='source',
    y='time_ms',
    color='source',
    title='⏱️ 数据加载耗时分布（箱线图）',
    labels={'time_ms': '耗时 (ms)', 'source': '数据源'},
    points='all',  # 显示所有散点（可关闭）
    color_discrete_sequence=px.colors.qualitative.Set2
)

# 添加平均耗时标注（交互式）
for source in data_sources:
    mean_time = np.mean(load_times[source])
    fig.add_annotation(
        x=source,
        y=mean_time,
        text=f'平均:{mean_time:.1f}ms',
        showarrow=True,
        arrowhead=2,
        arrowsize=0.5,
        arrowwidth=1,
        arrowcolor='gray',
        font=dict(size=9, color='gray')
    )

# 更新布局 + 交互式设置（对数坐标 + 悬停提示）
fig.update_layout(
    height=450,
    hovermode='x unified',
    xaxis_tickangle=-45,
    yaxis_type='log',  # 对数坐标（更适合耗时分布）
    yaxis_title='耗时 (ms, 对数坐标)',
    legend_title='数据源',
    showlegend=False  # 隐藏图例（颜色已区分）
)

# 更新悬停提示（显示统计信息）
fig.update_traces(
    hovertemplate='<b>%{x}</b><br>耗时：%{y:.1f}ms<br>四分位距：%{customdata}<extra></extra>',
    customdata=[f'{np.percentile(load_times[s], 75) - np.percentile(load_times[s], 25):.1f}ms' 
               for s in data_sources for _ in range(100)]
)

fig.show()

In [ ]:
# Plotly 吞吐量对比图（并行加速比 + 资源使用）
# 模拟不同并发数下的系统吞吐量（实际应通过压力测试获取）
concurrency_levels = [1, 2, 4, 8, 16, 32]
throughput_results = []

for conc in concurrency_levels:
    # 模拟吞吐量（考虑并行开销）
    base_throughput = 100  # 单线程基准吞吐量 (ops/s)
    overhead = 0.1 * np.log2(conc)  # 并行开销（对数增长）
    throughput = base_throughput * conc / (1 + overhead) + np.random.normal(0, 5)
    
    # 模拟内存使用（线性增长 + 随机波动）
    memory_usage = 200 + conc * 15 + np.random.normal(0, 10)
    
    throughput_results.append({
        'concurrency': conc,
        'throughput': throughput,
        'memory_mb': memory_usage,
        'speedup': throughput / base_throughput
    })

throughput_df = pd.DataFrame(throughput_results)

# 创建交互式吞吐量对比图（双轴 + 加速比标注）
fig = make_subplots(
    rows=1, cols=1,
    specs=[[{'secondary_y': True}]]
)

# 主轴：吞吐量（蓝色柱状图）
fig.add_trace(go.Bar(
    x=throughput_df['concurrency'].astype(str),
    y=throughput_df['throughput'],
    name='吞吐量',
    marker_color='skyblue',
    text=throughput_df['throughput'].apply(lambda x: f'{x:.0f}'),
    textposition='auto'
), secondary_y=False)

# 次轴：加速比（红色折线图）
fig.add_trace(go.Scatter(
    x=throughput_df['concurrency'],
    y=throughput_df['speedup'],
    name='加速比',
    mode='lines+markers',
    line=dict(color='red', width=3),
    marker=dict(size=8, symbol='diamond'),
    text=throughput_df['speedup'].apply(lambda x: f'{x:.1f}x'),
    textposition='top center'
), secondary_y=True)

# 添加理想加速比参考线（线性）
fig.add_trace(go.Scatter(
    x=throughput_df['concurrency'],
    y=throughput_df['concurrency'],
    name='理想加速比',
    mode='lines',
    line=dict(color='gray', width=1, dash='dash'),
    showlegend=True
), secondary_y=True)

# 更新布局 + 交互式设置（双轴 + 悬停提示）
fig.update_layout(
    title='⚡ 系统吞吐量与加速比分析',
    height=450,
    hovermode='x unified',
    legend=dict(orientation='h', yanchor='bottom', y=-0.2, xanchor='center', x=0.5)
)

# 更新坐标轴标签 + 双轴配置 + 对数坐标（并发数）
fig.update_xaxes(title_text='并发数', type='log', ticktext=['1','2','4','8','16','32'], tickvals=[1,2,4,8,16,32])
fig.update_yaxes(title_text='吞吐量 (ops/s)', secondary_y=False)
fig.update_yaxes(title_text='加速比 (x)', secondary_y=True, range=[0, 20])

fig.show()

## 3️⃣ 计算引擎性能 + 3D 资源使用可视化

In [ ]:
# 计算引擎性能测试（模拟）+ Plotly 3D 资源使用图（交互式）
# 模拟不同数据规模下的资源使用（实际应通过 profiling 获取）
data_sizes = [50, 100, 200, 500, 1000]  # 标的数量×天数
resource_usage = []

for size in data_sizes:
    # 模拟计算耗时（非线性增长）
    calc_time = 0.05 * size + 0.0001 * size**1.5 + np.random.normal(0, 0.02)
    
    # 模拟内存使用（线性 + 缓存开销）
    memory_mb = 50 + size * 0.8 + np.random.normal(0, 5)
    
    # 模拟 CPU 使用率（与数据规模正相关）
    cpu_percent = min(95, 20 + size * 0.08 + np.random.normal(0, 3))
    
    resource_usage.append({
        'data_size': size,
        'calc_time_s': calc_time,
        'memory_mb': memory_mb,
        'cpu_percent': cpu_percent
    })

resource_df = pd.DataFrame(resource_usage)

# 创建交互式 3D 资源使用图（数据规模×时间×内存）
fig = px.scatter_3d(
    resource_df,
    x='data_size',
    y='calc_time_s',
    z='memory_mb',
    color='cpu_percent',
    size='calc_time_s',
    title='🧮 计算引擎资源使用 3D 分析',
    labels={
        'data_size': '数据规模 (标的×天数)',
        'calc_time_s': '计算耗时 (秒)',
        'memory_mb': '内存使用 (MB)',
        'cpu_percent': 'CPU 使用率 (%)'
    },
    color_continuous_scale='Viridis',
    size_max=20
)

# 添加趋势拟合线（可选）
# 这里简化：添加一条参考平面（线性关系）
xx, yy = np.meshgrid(
    np.linspace(resource_df['data_size'].min(), resource_df['data_size'].max(), 10),
    np.linspace(resource_df['calc_time_s'].min(), resource_df['calc_time_s'].max(), 10)
)
zz = 50 + xx * 0.8 + yy * 10  # 简化平面方程（实际应拟合）

fig.add_trace(go.Surface(
    x=xx, y=yy, z=zz,
    opacity=0.2,
    showscale=False,
    name='参考平面',
    hoverinfo='skip'
))

# 更新布局 + 交互式设置（3D 旋转 + 悬停提示）
fig.update_layout(
    height=500,
    hovermode='closest',
    scene=dict(
        xaxis_title='数据规模',
        yaxis_title='计算耗时 (秒)',
        zaxis_title='内存使用 (MB)',
        camera=dict(
            eye=dict(x=1.5, y=1.5, z=1.5)  # 初始视角（可交互旋转）
        )
    ),
    coloraxis_colorbar=dict(title='CPU 使用率 (%)')
)

fig.show()

In [ ]:
# Plotly 性能指标汇总仪表板（多指标 + 交互式）
# 汇总关键性能指标（模拟）
perf_summary = pd.DataFrame([
    {'指标': '缓存命中率', '目标': '≥80%', '实际': '85.2%', '状态': '✅ 达标'},
    {'指标': '数据加载耗时', '目标': '≤500ms', '实际': '320ms', '状态': '✅ 达标'},
    {'指标': '价格计算耗时', '目标': '≤200ms', '实际': '145ms', '状态': '✅ 达标'},
    {'指标': '端到端吞吐量', '目标': '≥50 ops/s', '实际': '68 ops/s', '状态': '✅ 达标'},
    {'指标': '内存峰值', '目标': '≤500MB', '实际': '380MB', '状态': '✅ 达标'}
])

# 创建交互式性能汇总仪表板（表格 + 进度条组合）
fig = go.Figure(data=[go.Table(
    header=dict(
        values=[f'<b>{col}</b>' for col in perf_summary.columns],
        fill_color='royalblue',
        align='center',
        font=dict(color='white', size=11)
    ),
    cells=dict(
        values=[perf_summary[col] for col in perf_summary.columns],
        fill_color=[['lightgreen' if s=='✅ 达标' else 'lightyellow' 
                   for s in perf_summary['状态']]] * len(perf_summary.columns),
        align='center',
        font=dict(size=10),
        height=30
    )
)])

fig.update_layout(
    title='📋 性能基准测试总结',
    height=300,
    margin=dict(l=20, r=20, t=50, b=20)
)

fig.show()

# 导出为交互式性能报告（可选）
# fig.write_html('output/performance_test_report.html', include_plotlyjs='cdn')

## 📊 测试总结 + 导出交互式报告

In [ ]:
# 清理资源 + 最终状态输出 + Plotly 性能分析高级提示 + 优化建议
print("\n" + "="*60)
print("🎉 性能基准测试完成！")
print("📊 所有图表均为 Plotly 交互式可视化")
print("💡 性能分析高级提示：")
print("   • 箱线图：识别异常值 + 分布特征")
print("   • 热力图：快速定位性能瓶颈区域")
print("   • 3D 图表：多维度资源使用关联分析")
print("   • 双轴图：对比不同量纲的性能指标")
print("\n🔧 优化建议（基于测试结果）：")
print("   • 缓存容量建议：1000-2000 条（命中率>85%）")
print("   • 并发数建议：8-16（加速比接近线性）")
print("   • 数据加载优化：优先缓存命中 + 批量查询")
print("   • 内存优化：分批处理大数据集")
print("="*60)